# HW2_2

## 数据准备
1. 读取 `FinancialPhraseBank.csv`
2. 使用 `nltk` 分词，并进行小写化、去除特殊符号
3. 标签映射：`positive -> 1`，`negative -> -1`，`neutral -> 0`
4. 加载预训练词向量 `glove-wiki-gigaword-50.wordvectors`
5. 将句子转为向量序列（含 OOV 处理策略）
6. 按 `7 : 1.5 : 1.5` 划分 Train/Val/Test，并固定随机种子保证复现

In [1]:
# ===== 1) 导入依赖 =====
from pathlib import Path
from typing import List, Sequence
import re

import numpy as np
import pandas as pd
import nltk
from gensim.models import KeyedVectors
from sklearn.model_selection import train_test_split

In [2]:
# ===== 2) 全局配置（可按需修改） =====
CSV_PATH = Path("FinancialPhraseBank.csv")
EMBEDDING_PATH = Path("glove-wiki-gigaword-50.wordvectors")
OUTPUT_DIR = Path("prepared_data")

# 固定随机种子，保证数据划分与 OOV 随机向量可复现
RANDOM_SEED = 42

# 标签映射：文本标签 -> 数字标签
LABEL_MAP = {
    "positive": 1,
    "negative": -1,
    "neutral": 0,
}

# OOV 策略："zero"（统一零向量）或 "random"（固定种子的随机向量）
OOV_STRATEGY = "zero"

np.random.seed(RANDOM_SEED)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("配置完成。")
print(f"CSV_PATH = {CSV_PATH.resolve()}")
print(f"EMBEDDING_PATH = {EMBEDDING_PATH.resolve()}")
print(f"OOV_STRATEGY = {OOV_STRATEGY}")

配置完成。
CSV_PATH = D:\BigFiles\hw2\FinancialPhraseBank.csv
EMBEDDING_PATH = D:\BigFiles\hw2\glove-wiki-gigaword-50.wordvectors
OOV_STRATEGY = zero


In [ ]:
# ===== 3) 文本预处理函数（无网络下载依赖） =====
tokenizer = nltk.tokenize.WordPunctTokenizer()


def preprocess_text(text: str) -> List[str]:
    """
    预处理流程：
    1) 转为小写
    2) 使用正则仅保留字母/数字/空白（去除特殊符号）
    3) 使用 NLTK 分词
    4) 去掉空 token
    """
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    tokens = tokenizer.tokenize(text)
    tokens = [tok for tok in tokens if tok.strip()]
    return tokens

print("预处理函数定义完成（不需要下载 NLTK 资源）。")

预处理函数定义完成（不需要下载 NLTK 资源）。


In [4]:
# ===== 4) 读取数据 + 标签映射 =====
# 期望列：text, label
df = pd.read_csv(CSV_PATH)
required_cols = {"text", "label"}
if not required_cols.issubset(df.columns):
    raise ValueError(f"CSV 必须包含列 {required_cols}，当前列为 {set(df.columns)}")

# 应用文本预处理
df["tokens"] = df["text"].apply(preprocess_text)

# 标签转数字
df["label_num"] = df["label"].map(LABEL_MAP)
if df["label_num"].isna().any():
    unknown = sorted(df.loc[df["label_num"].isna(), "label"].unique())
    raise ValueError(f"发现未定义标签：{unknown}，请补充 LABEL_MAP")

df["label_num"] = df["label_num"].astype(int)

print("数据读取与预处理完成。")
print(f"样本总数: {len(df)}")
print(df[["text", "tokens", "label", "label_num"]].head(3))

数据读取与预处理完成。
样本总数: 4846
                                                text  \
0  According to Gran , the company has no plans t...   
1  Technopolis plans to develop in stages an area...   
2  The international electronic industry company ...   

                                              tokens     label  label_num  
0  [according, to, gran, the, company, has, no, p...   neutral          0  
1  [technopolis, plans, to, develop, in, stages, ...   neutral          0  
2  [the, international, electronic, industry, com...  negative         -1  


In [5]:
# ===== 5) 加载 GloVe 词向量 + 句子向量化 =====
# 优先按 gensim KeyedVectors 原生格式读取；若失败，可按 Word2Vec 文本格式回退
try:
    wv = KeyedVectors.load(str(EMBEDDING_PATH), mmap="r")
except Exception:
    wv = KeyedVectors.load_word2vec_format(str(EMBEDDING_PATH), binary=False)

embedding_dim = wv.vector_size
rng = np.random.default_rng(RANDOM_SEED)

# 为 OOV 词创建统一向量（zero 或 random）
if OOV_STRATEGY == "zero":
    oov_vector = np.zeros(embedding_dim, dtype=np.float32)
elif OOV_STRATEGY == "random":
    oov_vector = rng.normal(loc=0.0, scale=0.1, size=embedding_dim).astype(np.float32)
else:
    raise ValueError("OOV_STRATEGY 仅支持 'zero' 或 'random'")


def sentence_to_vector_sequence(tokens: Sequence[str]) -> np.ndarray:
    """
    将 token 列表映射为向量序列，返回形状：(句长, embedding_dim)
    若 token 不在词表中，使用统一 OOV 向量。
    """
    vectors = []
    for tok in tokens:
        if tok in wv:
            vectors.append(wv[tok])
        else:
            vectors.append(oov_vector)

    # 极端情况：空句子时返回 (0, embedding_dim)
    if len(vectors) == 0:
        return np.zeros((0, embedding_dim), dtype=np.float32)

    return np.asarray(vectors, dtype=np.float32)


df["vector_seq"] = df["tokens"].apply(sentence_to_vector_sequence)

print("词向量嵌入完成。")
print(f"embedding_dim = {embedding_dim}")
print(f"第一条样本向量序列形状: {df['vector_seq'].iloc[0].shape}")

词向量嵌入完成。
embedding_dim = 50
第一条样本向量序列形状: (22, 50)


In [6]:
# ===== 6) 按 7:1.5:1.5 划分 Train / Val / Test =====
# 第一步：Train = 70%，Temp = 30%
train_df, temp_df = train_test_split(
    df,
    test_size=0.30,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=df["label_num"],
)

# 第二步：Temp 再平分为 Val/Test，各 15%
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_SEED,
    shuffle=True,
    stratify=temp_df["label_num"],
)

print("划分完成：")
print(f"Train: {len(train_df)} ({len(train_df)/len(df):.2%})")
print(f"Val  : {len(val_df)} ({len(val_df)/len(df):.2%})")
print(f"Test : {len(test_df)} ({len(test_df)/len(df):.2%})")

划分完成：
Train: 3392 (70.00%)
Val  : 727 (15.00%)
Test : 727 (15.00%)


In [7]:
# ===== 7) 保存结果（便于后续训练直接读取） =====
# 这里使用 pickle 保留 object 列（tokens / vector_seq）
train_path = OUTPUT_DIR / "train.pkl"
val_path = OUTPUT_DIR / "val.pkl"
test_path = OUTPUT_DIR / "test.pkl"

train_df.to_pickle(train_path)
val_df.to_pickle(val_path)
test_df.to_pickle(test_path)

print("保存完成：")
print(train_path.resolve())
print(val_path.resolve())
print(test_path.resolve())

# 如需仅保存基础字段（不含向量序列）为 CSV，可取消以下注释：
# train_df[["text", "label", "label_num"]].to_csv(OUTPUT_DIR / "train.csv", index=False)
# val_df[["text", "label", "label_num"]].to_csv(OUTPUT_DIR / "val.csv", index=False)
# test_df[["text", "label", "label_num"]].to_csv(OUTPUT_DIR / "test.csv", index=False)

保存完成：
D:\BigFiles\hw2\prepared_data\train.pkl
D:\BigFiles\hw2\prepared_data\val.pkl
D:\BigFiles\hw2\prepared_data\test.pkl


## 模型构建与训练评估

1. 使用 `nn.LSTM + nn.Linear` 构建三分类模型
2. 训练 10 个 Epoch
3. 每个 Epoch 在验证集计算 Accuracy
4. 自动保存验证集 Accuracy 最佳的模型参数
5. 训练结束后加载最佳模型，在测试集计算 Accuracy 和 F1-score

对于句子长度不同的问题，使用 `torch.nn.utils.rnn.pad_sequence` 进行批次内的动态 padding，确保每个批次的输入维度一致。

In [8]:
# ===== 8) 构建数据集、DataLoader 与 LSTM 模型 =====
import random

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence
from sklearn.metrics import f1_score


# 固定随机种子，保证训练过程可复现
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


# 将标签从 -1/0/1 映射到 CrossEntropyLoss 需要的 0/1/2
label_to_index = {-1: 0, 0: 1, 1: 2}
index_to_label = {v: k for k, v in label_to_index.items()}


class FinancialSequenceDataset(Dataset):
    """保存句子向量序列和分类标签的数据集。"""

    def __init__(self, dataframe):
        self.sequences = dataframe["vector_seq"].tolist()
        self.labels = dataframe["label_num"].map(label_to_index).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return self.sequences[idx], self.labels[idx]


def collate_batch(batch):
    """
    将不同长度的句子向量序列补齐成一个 batch。
    返回：
    - padded_seqs: (batch, max_len, emb_dim)
    - lengths: 每个样本原始长度
    - labels: 分类标签索引
    """
    seq_tensors = []
    lengths = []
    labels = []

    for seq, label in batch:
        # 防御式处理：若出现空序列，至少保留 1 个全零时间步，避免 pack 报错
        if seq.shape[0] == 0:
            seq = np.zeros((1, embedding_dim), dtype=np.float32)

        tensor_seq = torch.tensor(seq, dtype=torch.float32)
        seq_tensors.append(tensor_seq)
        lengths.append(tensor_seq.size(0))
        labels.append(label)

    padded_seqs = pad_sequence(seq_tensors, batch_first=True)
    lengths = torch.tensor(lengths, dtype=torch.long)
    labels = torch.tensor(labels, dtype=torch.long)
    return padded_seqs, lengths, labels


train_dataset = FinancialSequenceDataset(train_df)
val_dataset = FinancialSequenceDataset(val_df)
test_dataset = FinancialSequenceDataset(test_df)

BATCH_SIZE = 64
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, collate_fn=collate_batch)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_batch)


class LSTMClassifier(nn.Module):
    """基于 LSTM 的文本三分类模型。"""

    def __init__(self, input_dim, hidden_dim=128, num_layers=1, num_classes=3, dropout=0.2):
        super().__init__()
        # num_layers=1 时 LSTM 内部 dropout 不生效，这里显式置 0
        lstm_dropout = dropout if num_layers > 1 else 0.0
        self.lstm = nn.LSTM(
            input_size=input_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            dropout=lstm_dropout,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_dim, num_classes)

    def forward(self, padded_seqs, lengths):
        packed = pack_padded_sequence(
            padded_seqs,
            lengths.cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (h_n, _) = self.lstm(packed)
        last_hidden = h_n[-1]
        logits = self.fc(self.dropout(last_hidden))
        return logits


device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = LSTMClassifier(input_dim=embedding_dim, hidden_dim=128, num_layers=1, num_classes=3, dropout=0.2).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print(f"device = {device}")
print("模型、数据集与 DataLoader 构建完成。")

device = cpu
模型、数据集与 DataLoader 构建完成。


In [9]:
# ===== 9) 训练 10 个 Epoch，并在验证集上评估 Accuracy =====
EPOCHS = 10
best_val_acc = -1.0
best_model_path = OUTPUT_DIR / "best_lstm_model.pt"


def evaluate_accuracy(model, data_loader):
    """在给定数据集上计算准确率，返回 (acc, y_true, y_pred)。"""
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for padded_seqs, lengths, labels in data_loader:
            padded_seqs = padded_seqs.to(device)
            lengths = lengths.to(device)
            labels = labels.to(device)

            logits = model(padded_seqs, lengths)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().tolist())
            all_labels.extend(labels.cpu().tolist())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)
    acc = (all_preds == all_labels).mean()
    return float(acc), all_labels, all_preds


for epoch in range(1, EPOCHS + 1):
    model.train()
    running_loss = 0.0

    for padded_seqs, lengths, labels in train_loader:
        padded_seqs = padded_seqs.to(device)
        lengths = lengths.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()
        logits = model(padded_seqs, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)

    train_loss = running_loss / len(train_dataset)
    val_acc, _, _ = evaluate_accuracy(model, val_loader)

    # 保存验证集表现最好的模型参数
    if val_acc > best_val_acc:
        best_val_acc = float(val_acc)
        torch.save(
            {
                "model_state_dict": model.state_dict(),
                "best_val_acc": float(best_val_acc),
                "epoch": int(epoch),
            },
            best_model_path,
        )

    print(f"Epoch [{epoch:02d}/{EPOCHS}] - Train Loss: {train_loss:.4f} - Val Acc: {val_acc:.4f} - Best Val Acc: {best_val_acc:.4f}")

print("训练完成。")
print(f"最佳模型文件: {best_model_path.resolve()}")

Epoch [01/10] - Train Loss: 0.9062 - Val Acc: 0.6382 - Best Val Acc: 0.6382
Epoch [02/10] - Train Loss: 0.7966 - Val Acc: 0.6740 - Best Val Acc: 0.6740
Epoch [03/10] - Train Loss: 0.7546 - Val Acc: 0.6534 - Best Val Acc: 0.6740
Epoch [04/10] - Train Loss: 0.7344 - Val Acc: 0.6575 - Best Val Acc: 0.6740
Epoch [05/10] - Train Loss: 0.7210 - Val Acc: 0.6671 - Best Val Acc: 0.6740
Epoch [06/10] - Train Loss: 0.6780 - Val Acc: 0.6575 - Best Val Acc: 0.6740
Epoch [07/10] - Train Loss: 0.6656 - Val Acc: 0.6699 - Best Val Acc: 0.6740
Epoch [08/10] - Train Loss: 0.6307 - Val Acc: 0.6836 - Best Val Acc: 0.6836
Epoch [09/10] - Train Loss: 0.6040 - Val Acc: 0.6781 - Best Val Acc: 0.6836
Epoch [10/10] - Train Loss: 0.5817 - Val Acc: 0.6781 - Best Val Acc: 0.6836
训练完成。
最佳模型文件: D:\BigFiles\hw2\prepared_data\best_lstm_model.pt


In [11]:
# ===== 10) 加载最优模型并在测试集计算 Accuracy 与 F1-score =====
# 说明：PyTorch 2.6+ 默认 weights_only=True，这里显式关闭以读取完整 checkpoint 字典。
checkpoint = torch.load(best_model_path, map_location=device, weights_only=False)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# 在测试集计算预测结果
test_acc, test_true_idx, test_pred_idx = evaluate_accuracy(model, test_loader)

# 将类别索引映射回原始标签（-1/0/1）再计算宏平均 F1
test_true_labels = np.array([index_to_label[i] for i in test_true_idx])
test_pred_labels = np.array([index_to_label[i] for i in test_pred_idx])
test_f1_macro = f1_score(test_true_labels, test_pred_labels, average="macro")

print("测试集评估结果：")
print(f"Best Val Acc (保存时): {checkpoint['best_val_acc']:.4f}")
print(f"Best Epoch          : {checkpoint['epoch']}")
print(f"Test Accuracy       : {test_acc:.4f}")
print(f"Test F1-score(macro): {test_f1_macro:.4f}")

测试集评估结果：
Best Val Acc (保存时): 0.6836
Best Epoch          : 8
Test Accuracy       : 0.6836
Test F1-score(macro): 0.5686
